In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy.stats import skew, kurtosis
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem, DataStructs
from itertools import combinations
from itertools import combinations
from collections import Counter
from mordred import Calculator, descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import pubchempy as pcp
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
plt.rcParams['font.sans-serif'] = ["Arial"]

In [ ]:
# 读取 CSV 文件
file_name = "../../data/source_data/property_models/DM.csv"  # installed external data
df = pd.read_csv(file_name)

# 确保 SMILES 列存在
smiles_column = "SMILES"
if smiles_column not in df.columns:
    raise ValueError(f"CSV 文件中未找到 '{smiles_column}' 列，请检查列名")

# 提取 SMILES，并去除空值
smiles_list = df[smiles_column].dropna().tolist()

# 初始化 Mordred 计算器
calc = Calculator(descriptors, ignore_3D=True)

# 获取描述符名称（直接通过map提取）
descriptor_names = list(map(lambda x: x, calc.descriptors))  # 这里直接提取元组中的名称

# 计算描述符（带进度条）
results = []
valid_smiles = []
for smiles in tqdm(smiles_list, desc="计算 Mordred 描述符", unit="分子"):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        descriptors_values = calc(mol)
        results.append(descriptors_values)
        valid_smiles.append(smiles)  # 记录有效的 SMILES

# 转换为 DataFrame
df_descriptors = pd.DataFrame(results)

# 将列名设置为描述符的名称
df_descriptors.columns = descriptor_names

# 将原始的 SMILES 列插入到数据框的第一列
df_descriptors.insert(0, "SMILES", valid_smiles)

# 保存为新的 CSV 文件
# output_file = "mordred_descriptors.csv"
# df_descriptors.to_csv(output_file, index=False)

# print(f"\n计算完成！结果已保存到 {output_file}")

In [ ]:
# 将所有列转换为数值，无法转换为数值的会被标记为 NaN
df_numeric = df_descriptors.apply(pd.to_numeric, errors='coerce')

# 将所有 NaN 填充为 0
df_numeric.fillna(0, inplace=True)

df_numeric['SMILES']=smiles_list

# 打印处理后的数据形状
print(f"最终处理后的数据形状: {df_numeric.shape}")

# 保存结果为新的 CSV 文件
# output_file = "filled_mordred_descriptors.csv"
# df_numeric.to_csv(output_file, index=False)

# print(f"\n计算完成！结果已保存到 {output_file}")

In [ ]:
# 删除全为0的列之前的列数
initial_columns = df_numeric.shape[1]

# 删除全为0的列
df_numeric = df_numeric.loc[:, (df_numeric != 0).any(axis=0)]

# 删除后的列数
final_columns = df_numeric.shape[1]

# 打印删除的列数
print(f"删除了 {initial_columns - final_columns} 列")

In [ ]:
# 计算相关性矩阵（忽略第一列 SMILES）
correlation_matrix = df_numeric.iloc[:, 1:].corr()

# 找出相关性大于0.8的列对
high_corr_var = set()
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            colname = correlation_matrix.columns[i]
            high_corr_var.add(colname)

# 删除高相关性的列（只保留一个）
df_filtered = df_numeric.drop(columns=high_corr_var)

# 计算最终相关性矩阵
final_corr_matrix = df_filtered.iloc[:, 1:].corr()

# 打印筛选后的数据形状
print(f"筛选后的数据形状: {df_filtered.shape}")

# 保存最终的 DataFrame 和相关性矩阵
# df_filtered.to_csv("df_filtered.csv", index=False)
# final_corr_matrix.to_csv("correlation_matrix_filtered.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import matplotlib.ticker as ticker

# 设置阈值
threshold = 0.025

# 选择需要归一化的列（忽略第一列 SMILES）
descriptors = df_filtered.iloc[:, 1:]

# MinMax 归一化
scaler = MinMaxScaler()
descriptors_normalized = scaler.fit_transform(descriptors)

# 将归一化后的数据转换为 DataFrame，保持列名
descriptors_normalized_df = pd.DataFrame(descriptors_normalized, columns=descriptors.columns)

# 计算每个描述符的方差
variances = descriptors_normalized_df.var()

# ---------------- 绘图部分 ----------------
plt.figure(figsize=(7.5, 6))
plt.hist(variances, bins=30, color='moccasin', edgecolor='black')
# plt.title('Variance Distribution of Descriptors', fontsize=20)
plt.xlabel('Standard deviation', fontsize=24)
plt.ylabel('Counts', fontsize=24)
plt.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
plt.tick_params(axis="both", direction="in", labelsize=20)
plt.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold = {threshold}')
plt.legend(fontsize=16)

# 设置 x 轴刻度格式为两位小数
ax = plt.gca()
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

# 加粗边框
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

plt.tight_layout()
plt.show()

# ---------------- 筛选方差高于阈值的描述符 ----------------
high_variance_columns = variances[variances >= threshold].index.tolist()

# 构建新的 DataFrame，保留 SMILES 和筛选后的描述符列
df_final = pd.concat([df_filtered.iloc[:, [0]], df_filtered[high_variance_columns]], axis=1)
df_final.to_csv("df_final.csv", index=False)

# 可选：打印信息
print(f"原始描述符数量：{descriptors.shape[1]}")
print(f"筛选后保留的描述符数量：{len(high_variance_columns)}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# 假设 variances 是你已计算的包含标准差的 Series，筛选出大于 0.025 的描述符
high_variance = variances[variances > 0.025]

# 排序和重置索引
sorted_data = pd.DataFrame({'Descriptors': high_variance.index, 'std_dev': high_variance.values})
sorted_data = sorted_data.sort_values(by='std_dev', ascending=False).reset_index(drop=True)

# 颜色映射
colors = cm.plasma(sorted_data.index / len(sorted_data))

# 绘图
plt.figure(figsize=(12, 8))
fontsize = 20
bars = plt.bar(sorted_data.index, sorted_data['std_dev'], color=colors)

# 字体和标签设置
plt.rc('xtick', labelsize=fontsize)
plt.rc('ytick', labelsize=fontsize)
plt.rcParams['font.family'] = "Arial"
plt.xlabel('Descriptor', fontsize=fontsize)
plt.ylabel('Standard deviation', fontsize=fontsize)
plt.tick_params(direction='in')

# 设置 y 轴刻度和范围
plt.ylim(0, 0.15)
plt.yticks(np.arange(0, 0.18, 0.04))

# 设置 x 轴刻度
x_ticks = sorted_data['Descriptors'][::5]
x_tick_positions = sorted_data.index[::5]
# plt.xticks(x_tick_positions, x_ticks, rotation=45, ha='right', fontsize=12)

# 减少左右边距
plt.xlim(-1.5, len(sorted_data) - 0.5)

# 加粗边框
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)

plt.show()


In [ ]:
x_tick_positions

In [ ]:
x_ticks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# 假设 variances 是你已计算的包含标准差的 Series，筛选出大于 0.025 的描述符
high_variance = variances[variances > 0.025]

# 排序和重置索引
sorted_data = pd.DataFrame({'Descriptors': high_variance.index, 'std_dev': high_variance.values})
sorted_data = sorted_data.sort_values(by='std_dev', ascending=False).reset_index(drop=True)

# 颜色映射
colors = cm.plasma(sorted_data.index / len(sorted_data))

# 绘图
plt.figure(figsize=(12, 8))
fontsize = 20
bars = plt.bar(sorted_data.index, sorted_data['std_dev'], color=colors)

# 字体和标签设置
plt.rc('xtick', labelsize=fontsize)
plt.rc('ytick', labelsize=fontsize)
plt.rcParams['font.family'] = "Arial"
plt.xlabel('Descriptor', fontsize=fontsize)
plt.ylabel('Standard deviation', fontsize=fontsize)
plt.tick_params(direction='in')

# 设置 y 轴刻度和范围
plt.ylim(0, 0.15)
plt.yticks(np.arange(0, 0.18, 0.04))

# 使用提供的 xticks 描述符
x_ticks = [
    "MAXsCH3", "nN", "RotRatio", "EState_VSA4", "JGI3", "AATS7d", 
    "PEOE_VSA2", "NssO", "PEOE_VSA1", "ATSC0c", "nHRing", "AATS4dv", 
    "naHRing", "ATSC1v", "GATS2s", "GATS3Z", "GATS3c", "ATSC1p", 
    "SIC1", "AATS0se", "GATS2c", "ATSC2c", "MATS3Z", "MATS5Z", "RPCG"
]

# 对应的 x_tick_positions 为索引位置
x_tick_positions = [
    0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 
    85, 90, 95, 100, 105, 110, 115, 120
]

# 设置 x 轴刻度
plt.xticks(x_tick_positions, x_ticks, rotation=45, ha='right', fontsize=12)

# 减少左右边距
plt.xlim(-1.5, len(sorted_data) - 0.5)

# 加粗边框
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)

plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 读取 CSV 文件
file_name = "../../data/source_data/property_models/DM.csv"  # installed external data
df = pd.read_csv(file_name)

# 确保 DM 列存在
dm_column = "DM"
if dm_column not in df.columns:
    raise ValueError(f"CSV 文件中未找到 '{dm_column}' 列，请检查列名")

# 绘图：美化版的 DM 分布直方图
plt.figure(figsize=(7.5, 6))
sns.histplot(df[dm_column], kde=True, color='#EB8763', bins=30)

# 设置标题和轴标签样式
# plt.title('DM Distribution', fontsize=20)
plt.xlabel('Dipole Moment (Debye)', fontsize=24)
plt.ylabel('Counts', fontsize=24)
plt.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
plt.tick_params(axis="both", direction="in", labelsize=20)
plt.xlim(-0.2, 8.7)

# 加粗边框，设置为黑色
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

# 显示图形
plt.tight_layout()
plt.show()

# 输出 DM 的统计指标
mean_val = df[dm_column].mean()
var_val = df[dm_column].var()
std_val = df[dm_column].std()

print(f"DM 平均值（Mean）: {mean_val:.2f}")
print(f"DM 方差（Variance）: {var_val:.2f}")
print(f"DM 标准差（Standard Deviation）: {std_val:.2f}")